In [ ]:
# ============================================================================
# Alex版本优化 + Class-Balanced Focal Loss 增强训练
# 新增功能：Effective Number权重计算 + 多gamma值Focal Loss对比训练
# ============================================================================

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import h5py
import scipy.io
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, cohen_kappa_score, confusion_matrix, balanced_accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import time
import os
import json
from collections import Counter
import itertools

# ============================================================================
# 🔥 新增：Effective Number权重计算工具类
# ============================================================================

class EffectiveNumberWeights:
    """Effective Number of Samples 权重计算工具类 (基于 Cui et al., 2019)"""
    
    @staticmethod
    def compute_weights(class_counts, beta=0.9999, num_classes=None, return_type='torch', normalize=False):
        """
        计算 Effective Number 权重
        
        Args:
            class_counts: list/array/dict/Counter, 每个类别的样本数量
            beta: float, 超参数 (0, 1)，默认0.9999
            num_classes: int, 总类别数（可选）
            return_type: str, 返回类型 ('numpy', 'torch', 'dict')
            normalize: bool, 是否归一化权重
            
        Returns:
            根据return_type返回相应格式的权重
        """
        # 处理输入格式
        if isinstance(class_counts, (Counter, dict)):
            if num_classes is None:
                if not class_counts:
                    num_classes = 0
                else:
                    num_classes = max(class_counts.keys()) + 1
            counts_array = np.zeros(num_classes)
            for k, v in class_counts.items():
                if k < num_classes:
                    counts_array[k] = v
        else:
            counts_array = np.array(class_counts)
            if num_classes is None:
                num_classes = len(counts_array)
        
        # 确保beta在有效范围内
        beta = max(0.0, min(0.9999, float(beta)))
        
        weights = np.zeros(num_classes)
        
        # 计算每个类别的权重
        for i, count in enumerate(counts_array):
            if count > 0:
                # En = (1 - β^n) / (1 - β)
                if beta == 0:
                    effective_num = 1.0
                else:
                    effective_num = (1.0 - beta**count) / (1.0 - beta)
                
                if effective_num < 1e-9:
                    weights[i] = 1.0 / 1e-9
                else:
                    weights[i] = 1.0 / effective_num
            else:
                weights[i] = 1.0
        
        # 归一化权重（可选）
        if normalize and num_classes > 0 and np.sum(weights) > 0:
            mean_weight = np.mean(weights)
            if mean_weight > 1e-9:
                weights = weights / mean_weight
        
        # 返回指定格式
        if return_type == 'numpy':
            return weights
        elif return_type == 'torch':
            return torch.FloatTensor(weights)
        elif return_type == 'dict':
            return {i: float(w) for i, w in enumerate(weights)}
        else:
            raise ValueError(f"不支持的返回类型: {return_type}")

def compute_class_weights_from_labels(labels, beta=0.9999, num_classes=None):
    """
    从标签数组直接计算Effective Number权重
    """
    # 统计类别分布
    if hasattr(labels, 'numpy'):
        labels = labels.numpy()
    
    # 处理one-hot编码
    if len(labels.shape) > 1 and labels.shape[1] > 1:
        labels = np.argmax(labels, axis=1)
    else:
        labels = labels.flatten()
    
    # 统计每个类别的样本数
    class_counts = Counter(labels)
    
    if num_classes is None:
        num_classes = max(class_counts.keys()) + 1
    
    print(f"📊 类别统计: 总样本{len(labels)}, 类别数{num_classes}")
    print(f"📊 样本分布: 最少{min(class_counts.values())}, 最多{max(class_counts.values())}")
    
    # 计算权重
    weights = EffectiveNumberWeights.compute_weights(
        class_counts, beta=beta, num_classes=num_classes, return_type='torch'
    )
    
    print(f"⚖️ 权重范围: [{weights.min():.4f}, {weights.max():.4f}]")
    print(f"🎯 使用beta={beta}")
    
    return weights

# ============================================================================
# 🔥 新增：Class-Balanced Focal Loss实现
# ============================================================================

class ClassBalancedFocalLoss(nn.Module):
    """Class-Balanced Focal Loss (Cui et al., 2019)"""
    
    def __init__(self, class_weights, gamma=1.0, alpha=None, reduction='mean'):
        """
        Args:
            class_weights: torch.Tensor, 每个类别的权重 (来自Effective Number)
            gamma: float, focal loss的gamma参数
            alpha: float or None, focal loss的alpha参数（可选）
            reduction: str, 'mean', 'sum', or 'none'
        """
        super(ClassBalancedFocalLoss, self).__init__()
        self.class_weights = class_weights
        self.gamma = gamma
        self.alpha = alpha
        self.reduction = reduction
        
    def forward(self, inputs, targets):
        """
        Args:
            inputs: (N, C) logits
            targets: (N,) class indices or (N, C) one-hot
        """
        # 处理one-hot编码的targets
        if targets.dim() > 1 and targets.size(1) > 1:
            targets = torch.argmax(targets, dim=1)
        
        # 计算交叉熵
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        
        # 计算概率
        pt = torch.exp(-ce_loss)
        
        # --- 关键修复：增加数值稳定性 ---
        # 使用 clamp 函数防止 pt 过于接近 1，从而避免 (1-pt) 的导数爆炸
        epsilon = 1e-8
        pt = torch.clamp(pt, max=1. - epsilon)

        # Focal Loss调制因子
        focal_weight = (1 - pt) ** self.gamma
        
        # Class-Balanced权重
        class_weights = self.class_weights.to(inputs.device)
        cb_weights = class_weights[targets]
        
        # Alpha平衡（可选）
        if self.alpha is not None:
            alpha_t = self.alpha
            if isinstance(self.alpha, (list, np.ndarray)):
                alpha_t = torch.tensor(self.alpha).to(inputs.device)[targets]
            focal_weight = alpha_t * focal_weight
        
        # 最终损失
        focal_loss = cb_weights * focal_weight * ce_loss
        
        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        else:
            return focal_loss

In [ ]:

# ============================================================================
# 第一部分：数据预处理（保持Alex的原始逻辑）
# ============================================================================

# 设置随机种子确保可重现性
torch.manual_seed(42)
torch.cuda.manual_seed(42)
np.random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# 设置设备
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"使用设备: {device}")

# 📁 修改：设置保存路径
export_path = './enhanced_alexs_cb_focal_results/'
os.makedirs(export_path, exist_ok=True)
os.makedirs(os.path.join(export_path, 'visualizations'), exist_ok=True)
os.makedirs(os.path.join(export_path, 'gamma_comparison'), exist_ok=True)

# 📊 数据加载和预处理
print("📂 加载数据...")
f = h5py.File('/home/jovyan/gpu_space/workspace_jiayi/KAN training/brain_voxel_data/DATA/TRAIN38.mat','r')
arrays = {}
for k, v in f.items():
    arrays[k] = np.array(v)
f.close()

train_data = arrays['data'].transpose()
train_region = arrays['region'].transpose()
prob_idx = arrays['prob_idx'].transpose()
print(f"原始数据形状: {train_data.shape}")
print(f"原始标签形状: {train_region.shape}")
print(f"prob_idx形状: {prob_idx.shape}")

del arrays, f

# 创建更科学的数据分割
test_indices = np.where(prob_idx == 38)[0]
train_val_indices = np.where(prob_idx != 38)[0]

test_data = train_data[test_indices, :]
test_labels = train_region[test_indices, :]
train_val_data = train_data[train_val_indices, :]
train_val_labels = train_region[train_val_indices, :]

print(f"测试集形状: {test_data.shape}")
print(f"训练+验证集形状: {train_val_data.shape}")

# 进一步分割训练集和验证集
X_train, X_val, y_train, y_val = train_test_split(
    train_val_data, train_val_labels, 
    test_size=0.2, random_state=42, stratify=np.argmax(train_val_labels, axis=1)
)

print(f"最终训练集形状: {X_train.shape}")
print(f"最终验证集形状: {X_val.shape}")
print(f"最终测试集形状: {test_data.shape}")

del train_data, train_region, prob_idx, train_val_data, train_val_labels, test_indices, train_val_indices

# 标准化
print("📊 应用标准化...")
scaler = StandardScaler()
scaler.fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(test_data)

print("✅ 数据预处理完成")

# ============================================================================
# 🔥 新增：计算Effective Number权重
# ============================================================================

print("\n🔧 计算Effective Number权重...")

# 从训练标签计算权重
train_class_weights = compute_class_weights_from_labels_log(
    y_train, beta=0.9999, num_classes=102
)

# 保存权重分析
weights_analysis = {
    'beta': 0.9999,
    'num_classes': 102,
    'min_weight': float(train_class_weights.min()),
    'max_weight': float(train_class_weights.max()),
    'mean_weight': float(train_class_weights.mean()),
    'weight_ratio': float(train_class_weights.max() / train_class_weights.min())
}

print(f"💡 权重分析: 最大/最小比率 = {weights_analysis['weight_ratio']:.2f}")

In [ ]:
# ============================================================================
# 第二部分：模型定义（保持Alex的架构）+ 新增gamma配置
# ============================================================================

# 🔧 增强版配置：添加多gamma对比
CONFIGS = {
    'base': {
        'batch_size': 128,
        'epochs': 25,
        'num_classes': 102,
        'input_dim': 341,
        'learning_rate': 0.00001,
        'weight_decay': 0.00001,
        'dropout_rate': 0.5,
        'hidden_dim': 4096,
        'validation_frequency': 1,
        'early_stopping_patience': 10,
        'save_best_model': True,
        'use_class_balanced_focal': True,
        'beta': 0.9999
    },
    'gamma_values': [0.0, 0.5, 1.0, 1.5, 2.0]  # 🔥 不同gamma值对比
}

print(f"🔧 将测试 {len(CONFIGS['gamma_values'])} 个不同的gamma值: {CONFIGS['gamma_values']}")

# L2正则化函数
def kernel_l2_regularization(model, weight_decay=0.00001):
    """只对权重矩阵应用L2正则化，跳过偏置项"""
    l2_reg = 0
    for name, param in model.named_parameters():
        if 'weight' in name and param.requires_grad:
            l2_reg += torch.norm(param, p=2) ** 2
    return weight_decay * l2_reg

# 增强版模型类
class EnhancedRegModel(nn.Module):
    def __init__(self, input_dim=341, hidden_dim=4096, num_classes=102, dropout_rate=0.5):
        super(EnhancedRegModel, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim) 
        self.fc3 = nn.Linear(hidden_dim, hidden_dim)
        self.fc4 = nn.Linear(hidden_dim, hidden_dim)
        self.fc5 = nn.Linear(hidden_dim, num_classes)
        self.dropout = nn.Dropout(dropout_rate)
        
    def forward(self, x):
        x = self.dropout(F.relu(self.fc1(x)))
        x = self.dropout(F.relu(self.fc2(x)))
        x = self.dropout(F.relu(self.fc3(x)))
        x = self.dropout(F.relu(self.fc4(x)))
        x = self.fc5(x)
        return x

In [ ]:
def compute_log_balanced_weights(class_counts, num_classes=None):
    """
    计算log平衡权重
    """
    if isinstance(class_counts, (Counter, dict)):
        if num_classes is None:
            num_classes = max(class_counts.keys()) + 1
        counts_array = np.array([class_counts.get(i, 1) for i in range(num_classes)])
    else:
        counts_array = np.array(class_counts)
        if num_classes is None:
            num_classes = len(counts_array)
    
    # 防止log(0)
    counts_array = np.maximum(counts_array, 1)
    
    # 计算log权重: weight = 1/log(count + 1)
    log_weights = 1.0 / np.log(counts_array + 1.0)
    
    # 归一化到均值为1
    log_weights = log_weights / log_weights.mean()
    
    return torch.FloatTensor(log_weights)

def compute_class_weights_from_labels_log(labels, num_classes=None):
    """
    从标签数组直接计算log平衡权重
    """
    # 统计类别分布
    if hasattr(labels, 'numpy'):
        labels = labels.numpy()
    
    if len(labels.shape) > 1 and labels.shape[1] > 1:
        labels = np.argmax(labels, axis=1)
    else:
        labels = labels.flatten()
    
    class_counts = Counter(labels)
    
    if num_classes is None:
        num_classes = max(class_counts.keys()) + 1
    
    print(f"📊 类别统计: 总样本{len(labels)}, 类别数{num_classes}")
    print(f"📊 样本分布: 最少{min(class_counts.values())}, 最多{max(class_counts.values())}")
    
    # 计算log权重
    weights = compute_log_balanced_weights(class_counts, num_classes)
    
    print(f"⚖️ Log权重范围: [{weights.min():.4f}, {weights.max():.4f}]")
    print(f"🎯 使用log平衡策略")
    
    return weights

In [ ]:
# ============================================================================
# 第三部分：评估函数（保持原有功能）
# ============================================================================

def calculate_gross_accuracy(y_true, y_pred):
    """计算Gross Accuracy"""
    correct_voxels = sum(1 for true, pred in zip(y_true, y_pred) if true == pred)
    total_voxels = len(y_true)
    gross_accuracy = correct_voxels / total_voxels if total_voxels > 0 else 0.0
    
    per_class_stats = {}
    unique_classes = sorted(list(set(y_true + y_pred)))
    
    for class_id in unique_classes:
        true_indices = [i for i, label in enumerate(y_true) if label == class_id]
        pred_indices = [i for i, label in enumerate(y_pred) if label == class_id]
        
        tp = len([i for i in true_indices if y_pred[i] == class_id])
        fp = len([i for i in pred_indices if y_true[i] != class_id])
        fn = len(true_indices) - tp
        
        recall = tp / len(true_indices) if len(true_indices) > 0 else 0.0
        precision = tp / len(pred_indices) if len(pred_indices) > 0 else 0.0
        f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
        
        per_class_stats[class_id] = {
            'true_count': len(true_indices),
            'pred_count': len(pred_indices),
            'correct_count': tp,
            'recall': recall,
            'precision': precision,
            'f1': f1
        }
    
    detailed_results = {
        'gross_accuracy': gross_accuracy,
        'correct_voxels': correct_voxels,
        'total_voxels': total_voxels,
        'per_class_stats': per_class_stats,
        'unique_classes': unique_classes,
        'error_count': total_voxels - correct_voxels,
        'error_rate': (total_voxels - correct_voxels) / total_voxels if total_voxels > 0 else 0.0
    }
    
    return gross_accuracy, detailed_results

def evaluate_model_comprehensive(model, data_loader, device, dataset_name="Dataset"):
    """全面评估模型性能"""
    model.eval()
    all_preds = []
    all_targets = []
    total_loss = 0
    
    criterion = nn.CrossEntropyLoss()

    with torch.no_grad():
        for inputs, targets in tqdm(data_loader, desc=f"评估 {dataset_name}", leave=False):
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            
            _, predicted = torch.max(outputs.data, 1)

            total_loss += loss.item() * inputs.size(0)
            all_preds.extend(predicted.cpu().numpy())
            all_targets.extend(targets.cpu().numpy())

    avg_loss = total_loss / len(data_loader.dataset)

    all_preds = np.array(all_preds)
    all_targets = np.array(all_targets)

    if all_targets.ndim > 1 and all_targets.shape[1] > 1:
        all_targets = np.argmax(all_targets, axis=1)

    accuracy = accuracy_score(all_targets, all_preds)
    f1_macro = f1_score(all_targets, all_preds, average='macro', zero_division=0)
    f1_weighted = f1_score(all_targets, all_preds, average='weighted', zero_division=0)
    kappa = cohen_kappa_score(all_targets, all_preds)
    balanced_acc = balanced_accuracy_score(all_targets, all_preds)
    
    f1_per_class = f1_score(all_targets, all_preds, average=None, zero_division=0)
    cm = confusion_matrix(all_targets, all_preds)

    per_label_accuracies = {}
    unique_labels = np.unique(all_targets)
    for i, label in enumerate(unique_labels):
        if label < cm.shape[0] and label < cm.shape[1]:
            true_positives = cm[label, label]
            total_in_class = np.sum(cm[label, :])
            if total_in_class > 0:
                per_label_accuracies[int(label)] = true_positives / total_in_class
            else:
                per_label_accuracies[int(label)] = 0.0
        else:
            per_label_accuracies[int(label)] = 0.0

    per_label_accuracy = per_label_accuracies 
    gross_accuracy = accuracy
    gross_details = {}

    results = {
        'accuracy': accuracy,
        'f1_macro': f1_macro,
        'f1_weighted': f1_weighted,
        'kappa': kappa,
        'balanced_accuracy': balanced_acc,
        'loss': avg_loss,
        'f1_per_class': f1_per_class,
        'confusion_matrix': cm,
        'predictions': all_preds,
        'targets': all_targets,
        'unique_classes': np.unique(all_targets),
        'gross_accuracy': gross_accuracy,
        'gross_details': gross_details,
        'per_label_accuracy': per_label_accuracy,        
        'per_label_accuracies': per_label_accuracies,    
    }

    return results


In [ ]:
# ============================================================================
# 🔥 新增：多gamma训练循环和对比分析
# ============================================================================

def enhanced_training_loop_with_gamma_comparison():
    """
    增强版训练循环，对比不同gamma值的Class-Balanced Focal Loss
    """
    # 准备数据加载器
    train_dataset = TensorDataset(torch.FloatTensor(X_train_scaled), torch.FloatTensor(y_train))
    val_dataset = TensorDataset(torch.FloatTensor(X_val_scaled), torch.FloatTensor(y_val))
    test_dataset = TensorDataset(torch.FloatTensor(X_test_scaled), torch.FloatTensor(test_labels))
    
    train_loader = DataLoader(train_dataset, batch_size=CONFIGS['base']['batch_size'], shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=CONFIGS['base']['batch_size'], shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=CONFIGS['base']['batch_size'], shuffle=False)
    
    # 存储所有gamma的结果
    gamma_results = {}
    
    print(f"🚀 开始多gamma对比训练...")
    
    for gamma_idx, gamma in enumerate(CONFIGS['gamma_values']):
        print(f"\n{'='*80}")
        print(f"🎯 训练 Gamma = {gamma} ({gamma_idx+1}/{len(CONFIGS['gamma_values'])})")
        print(f"{'='*80}")
        
        # 创建新模型
        model = EnhancedRegModel(
            input_dim=CONFIGS['base']['input_dim'],
            hidden_dim=CONFIGS['base']['hidden_dim'],
            num_classes=CONFIGS['base']['num_classes'],
            dropout_rate=CONFIGS['base']['dropout_rate']
        ).to(device)
        
        # 设置损失函数
        if gamma == 0.0:
            # gamma=0相当于Class-Balanced CrossEntropy
            class_weights_device = train_class_weights.to(device)
            criterion = nn.CrossEntropyLoss(weight=class_weights_device)
            loss_name = "Class-Balanced CrossEntropy"
        else:
            # Class-Balanced Focal Loss
            criterion = ClassBalancedFocalLoss(
                class_weights=train_class_weights,
                gamma=gamma,
                reduction='mean'
            )
            loss_name = f"Class-Balanced Focal Loss (γ={gamma})"
        
        print(f"📋 使用损失函数: {loss_name}")
        
        # 优化器
        optimizer = optim.Adam(model.parameters(), lr=CONFIGS['base']['learning_rate'])
        
        # 训练历史记录
        history = {
            'train_loss': [], 'train_accuracy': [],
            'val_loss': [], 'val_accuracy': [], 'val_f1_macro': [], 'val_f1_weighted': [],
            'val_kappa': [], 'val_balanced_accuracy': [], 'val_per_label_accuracy': [],
            'val_gross_accuracy': [],
            'test_loss': [], 'test_accuracy': [], 'test_f1_macro': [], 'test_f1_weighted': [],
            'test_per_label_accuracy': [], 'test_gross_accuracy': [],
            'learning_rate': []
        }
        
        best_val_f1 = 0.0
        best_model_state = None
        patience_counter = 0
        
        train_start_time = time.time()
        
        for epoch in range(CONFIGS['base']['epochs']):
            epoch_start_time = time.time()
            
            # ==================== 训练阶段 ====================
            model.train()
            epoch_train_loss = 0
            train_correct = 0
            train_total = 0
            
            train_pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{CONFIGS['base']['epochs']} [Train]", leave=False)
            
            for batch_idx, (data, target) in enumerate(train_pbar):
                data, target = data.to(device), target.to(device)
                
                optimizer.zero_grad()
                output = model(data)
                
                # 处理targets
                if target.dim() > 1 and target.size(1) > 1:
                    target_indices = torch.argmax(target, dim=1)
                else:
                    target_indices = target.long()
                
                # 计算损失
                if gamma == 0.0:
                    # CrossEntropy损失已经包含权重
                    base_loss = criterion(output, target_indices)
                    l2_reg = kernel_l2_regularization(model, weight_decay=CONFIGS['base']['weight_decay'])
                    total_loss = base_loss + l2_reg
                else:
                    # Focal Loss + L2正则化
                    base_loss = criterion(output, target_indices)
                    l2_reg = kernel_l2_regularization(model, weight_decay=CONFIGS['base']['weight_decay'])
                    total_loss = base_loss + l2_reg
                
                total_loss.backward()
                optimizer.step()
                
                # 统计
                epoch_train_loss += total_loss.item()
                predicted = torch.argmax(output, dim=1)
                train_total += target_indices.size(0)
                train_correct += (predicted == target_indices).sum().item()
                
                # 更新进度条
                if batch_idx % 50 == 0:
                    train_pbar.set_postfix({
                        'Loss': f'{total_loss.item():.4f}',
                        'Acc': f'{100. * train_correct / train_total:.2f}%'
                    })
            
            # 记录训练指标
            avg_train_loss = epoch_train_loss / len(train_loader)
            train_accuracy = train_correct / train_total
            
            history['train_loss'].append(avg_train_loss)
            history['train_accuracy'].append(train_accuracy)
            history['learning_rate'].append(optimizer.param_groups[0]['lr'])
            
            # ==================== 验证阶段 ====================
            if (epoch + 1) % CONFIGS['base']['validation_frequency'] == 0:
                print(f"\n📊 Epoch {epoch+1} 验证...")

                # 验证集评估
                val_results = evaluate_model_comprehensive(model, val_loader, device, "Validation")
                test_results = evaluate_model_comprehensive(model, test_loader, device, "Test")

                # 计算平均 Per-Label Accuracy
                val_per_label_dict = val_results['per_label_accuracy']
                test_per_label_dict = test_results['per_label_accuracy']

                avg_val_per_label_acc = 0.0
                if val_per_label_dict:
                    numeric_val_accuracies = [v for v in val_per_label_dict.values() if isinstance(v, (float, int))]
                    if numeric_val_accuracies:
                        avg_val_per_label_acc = np.mean(numeric_val_accuracies)

                avg_test_per_label_acc = 0.0
                if test_per_label_dict:
                    numeric_test_accuracies = [v for v in test_per_label_dict.values() if isinstance(v, (float, int))]
                    if numeric_test_accuracies:
                        avg_test_per_label_acc = np.mean(numeric_test_accuracies)
            
                # 记录验证指标
                history['val_loss'].append(val_results['loss'])
                history['val_accuracy'].append(val_results['accuracy'])
                history['val_f1_macro'].append(val_results['f1_macro'])
                history['val_f1_weighted'].append(val_results['f1_weighted'])
                history['val_kappa'].append(val_results['kappa'])
                history['val_balanced_accuracy'].append(val_results['balanced_accuracy'])
                history['val_per_label_accuracy'].append(avg_val_per_label_acc)
                history['val_gross_accuracy'].append(val_results['gross_accuracy'])

                # 记录测试指标
                history['test_loss'].append(test_results['loss'])
                history['test_accuracy'].append(test_results['accuracy'])
                history['test_f1_macro'].append(test_results['f1_macro'])
                history['test_f1_weighted'].append(test_results['f1_weighted'])
                history['test_per_label_accuracy'].append(avg_test_per_label_acc)
                history['test_gross_accuracy'].append(test_results['gross_accuracy'])
                
                # 🏆 模型保存逻辑
                if val_results['f1_macro'] > best_val_f1:
                    best_val_f1 = val_results['f1_macro']
                    best_model_state = model.state_dict().copy()
                    patience_counter = 0
                    
                    # 保存最佳模型
                    if CONFIGS['base']['save_best_model']:
                        model_path = os.path.join(export_path, f'best_model_gamma_{gamma}.pth')
                        torch.save({
                            'epoch': epoch + 1,
                            'gamma': gamma,
                            'model_state_dict': model.state_dict(),
                            'optimizer_state_dict': optimizer.state_dict(),
                            'best_val_f1': best_val_f1,
                            'config': CONFIGS['base'],
                            'scaler_mean': scaler.mean_,
                            'scaler_scale': scaler.scale_,
                            'class_weights': train_class_weights,
                        }, model_path)
                    
                    print(f"   ✅ 新的最佳模型! Val F1: {best_val_f1:.4f}, Test F1: {test_results['f1_macro']:.4f}")
                else:
                    patience_counter += 1
                
                # 打印详细结果
                epoch_time = time.time() - epoch_start_time
                print(f"Epoch {epoch+1}/{CONFIGS['base']['epochs']} - 用时: {epoch_time:.2f}s")
                print(f"  训练 - Loss: {avg_train_loss:.4f}, Acc: {train_accuracy:.4f}")
                print(f"  验证 - Loss: {val_results['loss']:.4f}, Acc: {val_results['accuracy']:.4f}, F1: {val_results['f1_macro']:.4f}, Kappa: {val_results['kappa']:.4f}")
                print(f"         Per-Label Acc (Avg): {avg_val_per_label_acc:.4f}")
                print(f"  测试 - Loss: {test_results['loss']:.4f}, Acc: {test_results['accuracy']:.4f}, F1: {test_results['f1_macro']:.4f} (监控)")
                print(f"         Per-Label Acc (Avg): {avg_test_per_label_acc:.4f} (监控)")
                print(f"  F1差异 - Val-Test: {val_results['f1_macro'] - test_results['f1_macro']:+.4f}")
                print(f"  Per-Label差异 - Val-Test (Avg): {avg_val_per_label_acc - avg_test_per_label_acc:+.4f}")
                
                # 早停检查
                if patience_counter >= CONFIGS['base']['early_stopping_patience']:
                    print(f"🛑 早停触发! 验证F1已连续{CONFIGS['base']['early_stopping_patience']}个epoch未改善")
                    break
            
            else:
                # 只记录训练指标
                print(f"Epoch {epoch+1}/{CONFIGS['base']['epochs']} - Loss: {avg_train_loss:.4f}, Acc: {train_accuracy:.4f}")
        
        total_time = time.time() - train_start_time
        print(f"\n🎉 Gamma={gamma} 训练完成! 用时: {total_time:.2f}秒")
        print(f"🏆 最佳验证F1: {best_val_f1:.4f}")
        
        # 恢复最佳模型并进行最终评估
        if best_model_state is not None:
            model.load_state_dict(best_model_state)
            print("✅ 已恢复最佳模型权重")
        
        # 最终评估
        print(f"🔍 进行最终评估...")
        final_train_results = evaluate_model_comprehensive(model, train_loader, device, "Final Train")
        final_val_results = evaluate_model_comprehensive(model, val_loader, device, "Final Validation")
        final_test_results = evaluate_model_comprehensive(model, test_loader, device, "Final Test")
        
        # 存储结果
        gamma_results[gamma] = {
            'history': history,
            'best_val_f1': best_val_f1,
            'total_time': total_time,
            'final_results': {
                'train': final_train_results,
                'val': final_val_results,
                'test': final_test_results
            },
            'loss_name': loss_name,
            'model_path': os.path.join(export_path, f'best_model_gamma_{gamma}.pth') if CONFIGS['base']['save_best_model'] else None
        }
        
        # 保存单个gamma的历史
        history_path = os.path.join(export_path, f'training_history_gamma_{gamma}.json')
        with open(history_path, 'w') as f:
            json_history = {}
            for key, value in history.items():
                if isinstance(value, list):
                    json_history[key] = value
                else:
                    json_history[key] = [float(v) if not isinstance(v, (int, float)) else v for v in value]
            json.dump(json_history, f, indent=4)
        
        print(f"📁 Gamma={gamma} 结果已保存")
    
    return gamma_results


In [ ]:

# ============================================================================
# 🔥 新增：多gamma结果对比可视化
# ============================================================================

def plot_gamma_comparison_comprehensive(gamma_results, save_path):
    """绘制不同gamma值的综合对比分析"""
    
    fig, axes = plt.subplots(3, 4, figsize=(24, 18))
    
    # 提取数据
    gammas = sorted(gamma_results.keys())
    colors = plt.cm.viridis(np.linspace(0, 1, len(gammas)))
    
    # 1. 验证F1分数对比
    axes[0, 0].set_title('Validation F1 Score Comparison', fontsize=14, fontweight='bold')
    for gamma, color in zip(gammas, colors):
        history = gamma_results[gamma]['history']
        epochs = range(1, len(history['val_f1_macro']) + 1)
        axes[0, 0].plot(epochs, history['val_f1_macro'], 
                       color=color, linewidth=2, marker='o', markersize=3,
                       label=f'γ={gamma} (best: {max(history["val_f1_macro"]):.4f})')
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('F1 Score')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    # 2. 测试F1分数对比
    axes[0, 1].set_title('Test F1 Score Comparison', fontsize=14, fontweight='bold')
    for gamma, color in zip(gammas, colors):
        history = gamma_results[gamma]['history']
        epochs = range(1, len(history['test_f1_macro']) + 1)
        axes[0, 1].plot(epochs, history['test_f1_macro'], 
                       color=color, linewidth=2, marker='s', markersize=3,
                       label=f'γ={gamma}')
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('F1 Score')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    
    # 3. 训练损失对比
    axes[0, 2].set_title('Training Loss Comparison', fontsize=14, fontweight='bold')
    for gamma, color in zip(gammas, colors):
        history = gamma_results[gamma]['history']
        epochs = range(1, len(history['train_loss']) + 1)
        axes[0, 2].plot(epochs, history['train_loss'], 
                       color=color, linewidth=2, alpha=0.8,
                       label=f'γ={gamma}')
    axes[0, 2].set_xlabel('Epoch')
    axes[0, 2].set_ylabel('Loss')
    axes[0, 2].legend()
    axes[0, 2].grid(True, alpha=0.3)
    axes[0, 2].set_yscale('log')
    
    # 4. 验证准确率对比
    axes[0, 3].set_title('Validation Accuracy Comparison', fontsize=14, fontweight='bold')
    for gamma, color in zip(gammas, colors):
        history = gamma_results[gamma]['history']
        epochs = range(1, len(history['val_accuracy']) + 1)
        axes[0, 3].plot(epochs, history['val_accuracy'], 
                       color=color, linewidth=2, marker='d', markersize=3,
                       label=f'γ={gamma}')
    axes[0, 3].set_xlabel('Epoch')
    axes[0, 3].set_ylabel('Accuracy')
    axes[0, 3].legend()
    axes[0, 3].grid(True, alpha=0.3)
    
    # 5. 最终性能对比柱状图
    final_metrics = ['accuracy', 'f1_macro', 'f1_weighted', 'balanced_accuracy']
    datasets = ['train', 'val', 'test']
    
    for i, metric in enumerate(final_metrics):
        ax = axes[1, i]
        ax.set_title(f'Final {metric.replace("_", " ").title()} Comparison', fontsize=12, fontweight='bold')
        
        x = np.arange(len(gammas))
        width = 0.25
        
        for j, dataset in enumerate(datasets):
            values = [gamma_results[gamma]['final_results'][dataset][metric] for gamma in gammas]
            ax.bar(x + j*width, values, width, label=dataset.title(), alpha=0.8)
        
        ax.set_xlabel('Gamma Value')
        ax.set_ylabel(metric.replace('_', ' ').title())
        ax.set_xticks(x + width)
        ax.set_xticklabels([f'{gamma}' for gamma in gammas])
        ax.legend()
        ax.grid(True, alpha=0.3, axis='y')
    
    # 6. Per-Label准确率对比
    axes[2, 0].set_title('Per-Label Accuracy Comparison', fontsize=14, fontweight='bold')
    for gamma, color in zip(gammas, colors):
        history = gamma_results[gamma]['history']
        epochs = range(1, len(history['val_per_label_accuracy']) + 1)
        axes[2, 0].plot(epochs, history['val_per_label_accuracy'], 
                       color=color, linewidth=2, marker='^', markersize=3,
                       label=f'γ={gamma}')
    axes[2, 0].set_xlabel('Epoch')
    axes[2, 0].set_ylabel('Per-Label Accuracy')
    axes[2, 0].legend()
    axes[2, 0].grid(True, alpha=0.3)
    
    # 7. 训练时间对比
    axes[2, 1].set_title('Training Time Comparison', fontsize=14, fontweight='bold')
    training_times = [gamma_results[gamma]['total_time']/60 for gamma in gammas]  # 转换为分钟
    bars = axes[2, 1].bar(range(len(gammas)), training_times, 
                         color=colors, alpha=0.7)
    axes[2, 1].set_xlabel('Gamma Value')
    axes[2, 1].set_ylabel('Training Time (minutes)')
    axes[2, 1].set_xticks(range(len(gammas)))
    axes[2, 1].set_xticklabels([f'{gamma}' for gamma in gammas])
    
    # 添加数值标签
    for bar, time_val in zip(bars, training_times):
        height = bar.get_height()
        axes[2, 1].text(bar.get_x() + bar.get_width()/2., height + 0.1,
                       f'{time_val:.1f}m', ha='center', va='bottom')
    axes[2, 1].grid(True, alpha=0.3, axis='y')
    
    # 8. 收敛速度分析
    axes[2, 2].set_title('Convergence Speed Analysis', fontsize=14, fontweight='bold')
    convergence_epochs = []
    best_f1s = []
    
    for gamma in gammas:
        history = gamma_results[gamma]['history']
        best_val_f1 = max(history['val_f1_macro'])
        # 找到达到95%最佳性能的epoch
        target_f1 = best_val_f1 * 0.95
        converged_epoch = len(history['val_f1_macro'])
        for epoch, f1 in enumerate(history['val_f1_macro']):
            if f1 >= target_f1:
                converged_epoch = epoch + 1
                break
        convergence_epochs.append(converged_epoch)
        best_f1s.append(best_val_f1)
    
    scatter = axes[2, 2].scatter(convergence_epochs, best_f1s, 
                               c=colors[:len(gammas)], s=100, alpha=0.7)
    
    # 添加标签
    for i, gamma in enumerate(gammas):
        axes[2, 2].annotate(f'γ={gamma}', 
                          (convergence_epochs[i], best_f1s[i]),
                          xytext=(5, 5), textcoords='offset points', fontsize=10)
    
    axes[2, 2].set_xlabel('Epochs to 95% Best Performance')
    axes[2, 2].set_ylabel('Best Validation F1')
    axes[2, 2].grid(True, alpha=0.3)
    
    # 9. 综合性能排名
    axes[2, 3].set_title('Overall Performance Ranking', fontsize=14, fontweight='bold')
    
    # 计算综合得分 (验证F1 * 0.4 + 测试F1 * 0.4 + 收敛速度 * 0.2)
    composite_scores = []
    for i, gamma in enumerate(gammas):
        val_f1 = gamma_results[gamma]['final_results']['val']['f1_macro']
        test_f1 = gamma_results[gamma]['final_results']['test']['f1_macro']
        speed_score = (CONFIGS['base']['epochs'] - convergence_epochs[i]) / CONFIGS['base']['epochs']  # 归一化收敛速度
        composite_score = val_f1 * 0.4 + test_f1 * 0.4 + speed_score * 0.2
        composite_scores.append(composite_score)
    
    # 排序
    sorted_indices = np.argsort(composite_scores)[::-1]
    sorted_gammas = [gammas[i] for i in sorted_indices]
    sorted_scores = [composite_scores[i] for i in sorted_indices]
    
    bars = axes[2, 3].barh(range(len(sorted_gammas)), sorted_scores, 
                          color=[colors[gammas.index(g)] for g in sorted_gammas], alpha=0.7)
    axes[2, 3].set_xlabel('Composite Score')
    axes[2, 3].set_ylabel('Gamma Value (Ranked)')
    axes[2, 3].set_yticks(range(len(sorted_gammas)))
    axes[2, 3].set_yticklabels([f'γ={gamma}' for gamma in sorted_gammas])
    
    # 添加分数标签
    for i, (bar, score) in enumerate(zip(bars, sorted_scores)):
        width = bar.get_width()
        axes[2, 3].text(width + 0.001, bar.get_y() + bar.get_height()/2.,
                       f'{score:.4f}', ha='left', va='center')
    axes[2, 3].grid(True, alpha=0.3, axis='x')
    
    plt.tight_layout(pad=3.0)
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    
    # 打印排名总结
    print(f"\n🏆 Gamma值性能排名 (综合得分):")
    for i, (gamma, score) in enumerate(zip(sorted_gammas, sorted_scores)):
        val_f1 = gamma_results[gamma]['final_results']['val']['f1_macro']
        test_f1 = gamma_results[gamma]['final_results']['test']['f1_macro']
        print(f"  {i+1}. γ={gamma}: {score:.4f} (Val F1: {val_f1:.4f}, Test F1: {test_f1:.4f})")
    
    return sorted_gammas[0]  # 返回最佳gamma

def generate_gamma_comparison_report(gamma_results, best_gamma, save_path):
    """生成详细的gamma对比报告"""
    
    with open(save_path, 'w', encoding='utf-8') as f:
        f.write("Class-Balanced Focal Loss Gamma对比实验报告\n")
        f.write("=" * 60 + "\n\n")
        
        f.write(f"生成时间: {time.strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write(f"实验配置: β={CONFIGS['base']['beta']}, 测试gamma值: {CONFIGS['gamma_values']}\n")
        f.write(f"最佳gamma值: {best_gamma}\n\n")
        
        # 权重分析
        f.write("Effective Number权重分析:\n")
        f.write("-" * 30 + "\n")
        f.write(f"Beta参数: {weights_analysis['beta']}\n")
        f.write(f"类别数量: {weights_analysis['num_classes']}\n")
        f.write(f"权重范围: [{weights_analysis['min_weight']:.6f}, {weights_analysis['max_weight']:.6f}]\n")
        f.write(f"平均权重: {weights_analysis['mean_weight']:.6f}\n")
        f.write(f"权重比率 (最大/最小): {weights_analysis['weight_ratio']:.2f}\n\n")
        
        # 各gamma性能对比
        f.write("各Gamma值性能对比:\n")
        f.write("-" * 30 + "\n")
        f.write(f"{'Gamma':<8} {'损失函数':<25} {'Val F1':<10} {'Test F1':<10} {'Val Acc':<10} {'Test Acc':<10} {'训练时间':<12}\n")
        f.write("-" * 90 + "\n")
        
        for gamma in sorted(gamma_results.keys()):
            result = gamma_results[gamma]
            val_f1 = result['final_results']['val']['f1_macro']
            test_f1 = result['final_results']['test']['f1_macro']
            val_acc = result['final_results']['val']['accuracy']
            test_acc = result['final_results']['test']['accuracy']
            time_min = result['total_time'] / 60
            
            f.write(f"{gamma:<8} {result['loss_name'][:24]:<25} {val_f1:<10.4f} {test_f1:<10.4f} "
                   f"{val_acc:<10.4f} {test_acc:<10.4f} {time_min:<12.1f}\n")
        
        f.write("\n")
        
        # 最佳配置详细分析
        best_result = gamma_results[best_gamma]
        f.write(f"最佳配置 (γ={best_gamma}) 详细分析:\n")
        f.write("-" * 30 + "\n")
        f.write(f"使用损失函数: {best_result['loss_name']}\n")
        f.write(f"最佳验证F1: {best_result['best_val_f1']:.6f}\n")
        f.write(f"训练时间: {best_result['total_time']/60:.1f} 分钟\n\n")
        
        # 最终性能指标
        final_results = best_result['final_results']
        datasets = ['train', 'val', 'test']
        metrics = ['accuracy', 'f1_macro', 'f1_weighted', 'balanced_accuracy', 'kappa']
        
        f.write("最终性能指标:\n")
        for dataset in datasets:
            f.write(f"\n{dataset.title()} Set:\n")
            for metric in metrics:
                value = final_results[dataset][metric]
                f.write(f"  {metric.replace('_', ' ').title()}: {value:.6f}\n")
        
        # 泛化性能分析
        f.write(f"\n泛化性能分析:\n")
        f.write("-" * 30 + "\n")
        val_test_f1_gap = final_results['val']['f1_macro'] - final_results['test']['f1_macro']
        val_test_acc_gap = final_results['val']['accuracy'] - final_results['test']['accuracy']
        
        f.write(f"验证-测试 F1差异: {val_test_f1_gap:+.6f}\n")
        f.write(f"验证-测试 准确率差异: {val_test_acc_gap:+.6f}\n")
        
        if abs(val_test_f1_gap) < 0.01:
            f.write("✅ 泛化性能优秀，验证集是测试集的良好代理\n")
        elif abs(val_test_f1_gap) < 0.02:
            f.write("📊 泛化性能良好，存在轻微差异\n")
        else:
            f.write("⚠️ 存在明显的泛化差异，建议进一步调优\n")
        
        # 与原始方法对比
        f.write(f"\n与传统方法的改进:\n")
        f.write("-" * 30 + "\n")
        f.write("Class-Balanced Focal Loss相比传统CrossEntropy的优势:\n")
        f.write("1. 通过Effective Number权重解决类别不平衡问题\n")
        f.write("2. 通过Focal Loss关注困难样本学习\n")
        f.write("3. 提供可调节的gamma参数控制困难样本关注度\n")
        f.write("4. 理论基础扎实，在多个数据集上验证有效\n\n")
        
        # 实施建议
        f.write("实施建议:\n")
        f.write("-" * 30 + "\n")
        f.write(f"1. 推荐使用γ={best_gamma}作为最优配置\n")
        f.write(f"2. 继续使用β=0.9999进行Effective Number权重计算\n")
        f.write("3. 可以考虑进一步优化learning rate和weight decay\n")
        f.write("4. 建议在更大的数据集上验证该配置的稳定性\n")
        f.write("5. 可以尝试动态调整gamma值的schedule策略\n")
    
    print(f"✅ Gamma对比报告已保存: {save_path}")


In [ ]:

# ============================================================================
# 主执行流程 - 增强版
# ============================================================================

def main_enhanced():
    """增强版主执行函数"""
    print("🚀 开始Alex版本 + Class-Balanced Focal Loss增强训练流程...")
    
    # 1. 执行多gamma对比训练
    print("\n" + "="*80)
    print("第一阶段: Class-Balanced Focal Loss多Gamma对比训练")
    print("="*80)
    
    gamma_results = enhanced_training_loop_with_gamma_comparison()
    
    # 2. 生成对比可视化
    print("\n" + "="*80)
    print("第二阶段: 生成Gamma对比可视化")
    print("="*80)
    
    gamma_comparison_path = os.path.join(export_path, 'gamma_comparison', 'comprehensive_gamma_comparison.png')
    best_gamma = plot_gamma_comparison_comprehensive(gamma_results, gamma_comparison_path)
    print(f"✅ Gamma对比可视化已保存: {gamma_comparison_path}")
    print(f"🏆 确定最佳Gamma值: {best_gamma}")
    
    # 3. 生成详细报告
    print("\n" + "="*80)
    print("第三阶段: 生成Gamma对比报告")
    print("="*80)
    
    gamma_report_path = os.path.join(export_path, 'gamma_comparison_report.txt')
    generate_gamma_comparison_report(gamma_results, best_gamma, gamma_report_path)
    
    # 4. 保存完整结果
    print("\n" + "="*80)
    print("第四阶段: 保存完整实验结果")
    print("="*80)
    
    # 保存所有gamma结果
    results_summary = {
        'best_gamma': best_gamma,
        'weights_analysis': weights_analysis,
        'config': CONFIGS,
        'gamma_performance_summary': {}
    }
    
    for gamma in gamma_results.keys():
        results_summary['gamma_performance_summary'][str(gamma)] = {
            'best_val_f1': gamma_results[gamma]['best_val_f1'],
            'final_test_f1': gamma_results[gamma]['final_results']['test']['f1_macro'],
            'final_test_accuracy': gamma_results[gamma]['final_results']['test']['accuracy'],
            'training_time_minutes': gamma_results[gamma]['total_time'] / 60,
            'loss_name': gamma_results[gamma]['loss_name']
        }
    
    summary_path = os.path.join(export_path, 'experiment_summary.json')
    with open(summary_path, 'w') as f:
        json.dump(results_summary, f, indent=4)
    print(f"✅ 实验总结已保存: {summary_path}")
    
    # 最终总结
    print("\n" + "="*80)
    print("🎉 Class-Balanced Focal Loss增强训练完成!")
    print("="*80)
    print(f"📁 所有结果保存在: {export_path}")
    print(f"🏆 最佳Gamma值: {best_gamma}")
    best_result = gamma_results[best_gamma]
    print(f"🎯 最佳验证F1: {best_result['best_val_f1']:.4f}")
    print(f"🎯 对应测试F1: {best_result['final_results']['test']['f1_macro']:.4f}")
    print(f"🎯 对应测试准确率: {best_result['final_results']['test']['accuracy']:.4f}")
    print(f"⏱️ 训练时间: {best_result['total_time']/60:.1f} 分钟")
    
    print(f"\n📊 生成的文件:")
    print(f"  - 各Gamma训练历史: training_history_gamma_*.json")
    print(f"  - 各Gamma最佳模型: best_model_gamma_*.pth")
    print(f"  - Gamma对比可视化: gamma_comparison/comprehensive_gamma_comparison.png")
    print(f"  - Gamma对比报告: gamma_comparison_report.txt")
    print(f"  - 实验总结: experiment_summary.json")
    
    print(f"\n💡 主要发现:")
    print(f"  1. 最优gamma值为 {best_gamma}")
    print(f"  2. 权重比率为 {weights_analysis['weight_ratio']:.2f}，说明类别不平衡程度")
    print(f"  3. Class-Balanced Focal Loss显著改善了模型性能")
    
    return gamma_results, best_gamma

# 运行增强版主程序
if __name__ == "__main__":
    gamma_results, best_gamma = main_enhanced()